# v7 - Validacao cruzada (5-fold) + TTA

Caracterizacao final do modelo v7 (encoder MedSigLIP **congelado** + cabeca, corte NEV=1400).
Nao muda a arquitetura — solidifica o numero e adiciona test-time augmentation.

**Metodologia:**
- 5-fold **estratificado** sobre os 1011 casos do Derm7pt (cada caso testado 1x).
- Cada fold treina a cabeca em (4/5 Derm7pt + HAM cortado), testa no 1/5 Derm7pt de fora.
- Encoder congelado + **cache de embeddings** -> treino da cabeca em segundos por fold.
- **TTA**: 4 views aumentadas + original, media das probabilidades (softmax).
- Reporta **media ± desvio** em 5 folds + **matriz de confusao agregada** (todos os 1011 casos).

Baseline single-split do v7: macro-F1 derm_test = 0,612 | accuracy = 0,737.

In [ ]:
import os, sys, site, importlib, subprocess
REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches(); site.main()
print("Setup OK")

In [ ]:
import copy
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptUnifiedDataset, HAM10000Dataset, classification_collate_fn,
    GROUP_TO_LABEL, LABEL_TO_GROUP, METADATA_DIM_V5,
    HAM_DX_TO_GROUP, ham10000_train_val_split,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights
from melanoma_tcc.utils.metrics import plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7pt-multimodal/release_v0"
DERM_META = f"{DERM7PT_DIR}/meta/meta.csv"
DERM_IMAGES = f"{DERM7PT_DIR}/images"
HAM_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
HAM_META = f"{HAM_DIR}/HAM10000_metadata.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEL_LABEL = GROUP_TO_LABEL['MEL']
TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]
print(f"Device: {device} | MEL label = {MEL_LABEL}")

In [ ]:
# Modelo v7: encoder CONGELADO + cabeca. Sem unfreeze, sem AMP (encoder nao treina).
model, processor = build_dermclassifier(hf_token=HF_TOKEN, num_classes=5,
                                        metadata_dim=METADATA_DIM_V5, freeze_vision=True)
model = model.to(device)
model.vision_encoder = model.vision_encoder.to(device)

# Guarda o estado INICIAL da cabeca para resetar a cada fold (mesmo init, dados diferentes)
init_head_state = {
    'vision_proj': copy.deepcopy(model.vision_proj.state_dict()),
    'metadata_encoder': copy.deepcopy(model.metadata_encoder.state_dict()),
    'classifier': copy.deepcopy(model.classifier.state_dict()),
}
def reset_head():
    model.vision_proj.load_state_dict(init_head_state['vision_proj'])
    model.metadata_encoder.load_state_dict(init_head_state['metadata_encoder'])
    model.classifier.load_state_dict(init_head_state['classifier'])

head_params = (list(model.vision_proj.parameters())
               + list(model.metadata_encoder.parameters())
               + list(model.classifier.parameters()))

def head_forward(emb, md):
    v = model.vision_proj(emb.to(device))
    m = model.metadata_encoder(md.float().to(device))
    return model.classifier(torch.cat([v, m], dim=-1))

print(f"Head params (treinaveis por fold): {sum(p.numel() for p in head_params):,}")

In [ ]:
# ===== Datasets: Derm7pt COMPLETO (1011) + HAM train cortado (NEV=1400) =====
NEV_TARGET = 1400

derm_full = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                  indexes_csv=None, augment=False)          # 1011, sem aug (p/ cache)
derm_full_aug = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                      indexes_csv=None, augment=True, seed=123)  # p/ TTA

ham_train_df, _ = ham10000_train_val_split(HAM_META, val_ratio=0.15, seed=42, filter_unknown=True)
ham_train_df = ham_train_df.copy()
ham_train_df['group'] = (ham_train_df['dx'].str.strip().str.lower().map(HAM_DX_TO_GROUP).fillna('MISC'))
nev_mask = ham_train_df['group'] == 'NEV'
if int(nev_mask.sum()) > NEV_TARGET:
    keep = ham_train_df[nev_mask].sample(n=NEV_TARGET, random_state=42)
    ham_train_df = pd.concat([keep, ham_train_df[~nev_mask]]).reset_index(drop=True)
ham_train = HAM10000Dataset(ham_train_df, HAM_DIR, processor, augment=False)

print(f"Derm7pt total: {len(derm_full)} | HAM train (cortado): {len(ham_train)}")
print("Derm7pt dist:", dict(Counter(derm_full.df['group'])))
print("HAM train dist:", dict(Counter(ham_train.df['group'])))

In [ ]:
# ===== Pre-computa embeddings UMA vez (encoder congelado) =====
@torch.no_grad()
def encode_dataset(ds, bs=32, workers=4):
    model.eval()
    loader = DataLoader(ds, batch_size=bs, shuffle=False,
                        collate_fn=classification_collate_fn, num_workers=workers)
    E, M, L = [], [], []
    for b in loader:
        E.append(model.encode_image(b['pixel_values'].to(device)).cpu())
        M.append(b['metadata']); L.append(b['labels'])
    return torch.cat(E), torch.cat(M), torch.cat(L)

print("Cacheando Derm7pt (1011)...")
derm_emb, derm_md, derm_lb = encode_dataset(derm_full)
print("Cacheando HAM train...")
ham_emb, ham_md, ham_lb = encode_dataset(ham_train)
derm_y = derm_lb.numpy()
print(f"Derm emb: {tuple(derm_emb.shape)} | HAM emb: {tuple(ham_emb.shape)}")

In [ ]:
# ===== Helpers de treino/avaliacao da cabeca =====
EPOCHS_FOLD = 25   # onde o v7 convergiu (~ep24); orcamento fixo, reprodutivel

def train_head(tr_emb, tr_md, tr_lb):
    reset_head()
    counts = [max(1, int((tr_lb == i).sum())) for i in range(5)]
    alpha = compute_class_weights(counts, mode="inverse_sqrt")
    criterion = FocalLoss(alpha=alpha, gamma=2.0, label_smoothing=0.05)
    opt = AdamW(head_params, lr=5e-4, weight_decay=0.01)
    sched = CosineAnnealingLR(opt, T_max=EPOCHS_FOLD)
    loader = DataLoader(TensorDataset(tr_emb, tr_md, tr_lb), batch_size=32, shuffle=True)
    for ep in range(EPOCHS_FOLD):
        model.train()
        for emb, md, lb in loader:
            lb = lb.to(device)
            loss = criterion(head_forward(emb, md), lb)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(head_params, 1.0); opt.step()
        sched.step()

@torch.no_grad()
def predict_plain(te_emb, te_md):
    model.eval()
    return F.softmax(head_forward(te_emb, te_md).float(), dim=-1).cpu()

@torch.no_grad()
def predict_tta(te_idx, n_aug=4):
    model.eval()
    prob = predict_plain(derm_emb[te_idx], derm_md[te_idx])          # view original (cache)
    sub = Subset(derm_full_aug, list(te_idx))
    for _ in range(n_aug):                                            # views aumentadas
        loader = DataLoader(sub, batch_size=32, shuffle=False,
                            collate_fn=classification_collate_fn, num_workers=0)
        E, M = [], []
        for b in loader:
            E.append(model.encode_image(b['pixel_values'].to(device)).cpu())
            M.append(b['metadata'])
        prob = prob + predict_plain(torch.cat(E), torch.cat(M))
    return (prob / (n_aug + 1)).argmax(dim=-1).numpy()

def metrics(y, p):
    return {
        'acc': float((y == p).mean()),
        'macro_f1': f1_score(y, p, average='macro'),
        'mel_rec': recall_score(y, p, labels=[MEL_LABEL], average='macro', zero_division=0),
        'mel_prec': precision_score(y, p, labels=[MEL_LABEL], average='macro', zero_division=0),
        'f1_perclass': f1_score(y, p, average=None, labels=list(range(5)), zero_division=0),
    }

In [ ]:
# ===== Loop 5-fold estratificado =====
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
res_plain, res_tta = [], []
oof_y, oof_plain, oof_tta = [], [], []   # predicoes out-of-fold (cobrem todos os 1011 casos)

for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(derm_y)), derm_y), 1):
    tr_emb = torch.cat([derm_emb[tr_idx], ham_emb])
    tr_md = torch.cat([derm_md[tr_idx], ham_md])
    tr_lb = torch.cat([derm_lb[tr_idx], ham_lb])
    train_head(tr_emb, tr_md, tr_lb)

    y = derm_lb[te_idx].numpy()
    p_plain = predict_plain(derm_emb[te_idx], derm_md[te_idx]).argmax(dim=-1).numpy()
    p_tta = predict_tta(te_idx, n_aug=4)
    oof_y.extend(y.tolist()); oof_plain.extend(p_plain.tolist()); oof_tta.extend(p_tta.tolist())
    mp, mt = metrics(y, p_plain), metrics(y, p_tta)
    res_plain.append(mp); res_tta.append(mt)
    print(f"Fold {fold}: [plain] acc={mp['acc']:.3f} macroF1={mp['macro_f1']:.3f} MELrec={mp['mel_rec']:.3f}"
          f"  |  [TTA] acc={mt['acc']:.3f} macroF1={mt['macro_f1']:.3f} MELrec={mt['mel_rec']:.3f}")

print("\nFolds concluidos.")

In [ ]:
# ===== Agregacao: media +/- desvio em 5 folds =====
def summarize(name, res):
    accs = np.array([r['acc'] for r in res])
    f1s = np.array([r['macro_f1'] for r in res])
    recs = np.array([r['mel_rec'] for r in res])
    precs = np.array([r['mel_prec'] for r in res])
    pc = np.array([r['f1_perclass'] for r in res])   # [5 folds, 5 classes]
    print(f"\n===== {name} (5-fold, media ± desvio) =====")
    print(f"accuracy  : {accs.mean():.4f} ± {accs.std():.4f}")
    print(f"macro-F1  : {f1s.mean():.4f} ± {f1s.std():.4f}")
    print(f"MEL recall: {recs.mean():.4f} ± {recs.std():.4f}")
    print(f"MEL prec  : {precs.mean():.4f} ± {precs.std():.4f}")
    print("F1 por classe:")
    for i, c in enumerate(TARGET_NAMES):
        print(f"    {c:5s}: {pc[:, i].mean():.4f} ± {pc[:, i].std():.4f}")

summarize("PLAIN (sem TTA)", res_plain)
summarize("TTA (5 views)", res_tta)

d_f1 = np.mean([r['macro_f1'] for r in res_tta]) - np.mean([r['macro_f1'] for r in res_plain])
print(f"\nGanho do TTA no macro-F1: {d_f1:+.4f}")
print("Baseline v7 single-split: macro-F1=0.612 | accuracy=0.737")

os.makedirs("/kaggle/working/derm-classifier-v7-kfold", exist_ok=True)
import json
with open("/kaggle/working/derm-classifier-v7-kfold/kfold_results.json", "w") as f:
    json.dump({'plain': [{k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()} for r in res_plain],
               'tta': [{k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()} for r in res_tta]},
              f, indent=2)
print("\nSalvou kfold_results.json")

In [ ]:
# ===== Matriz de confusao AGREGADA (out-of-fold: todos os 1011 casos) =====
# Como cada caso do Derm7pt eh testado exatamente 1x, juntar as predicoes dos 5 folds
# da uma matriz de confusao sobre o dataset inteiro (mais robusta que o single-split).
oof_y = np.array(oof_y); oof_plain = np.array(oof_plain); oof_tta = np.array(oof_tta)

print("=" * 60)
print(f"MATRIZ AGREGADA 5-FOLD — SEM TTA ({len(oof_y)} casos)")
print("=" * 60)
print(classification_report(oof_y, oof_plain, target_names=TARGET_NAMES, digits=4, zero_division=0))
plot_confusion_matrix(oof_y, oof_plain, target_names=TARGET_NAMES,
                      save_path='/kaggle/working/derm-classifier-v7-kfold-cm.png')

print("\n" + "=" * 60)
print(f"MATRIZ AGREGADA 5-FOLD — COM TTA ({len(oof_y)} casos)")
print("=" * 60)
print(classification_report(oof_y, oof_tta, target_names=TARGET_NAMES, digits=4, zero_division=0))
plot_confusion_matrix(oof_y, oof_tta, target_names=TARGET_NAMES,
                      save_path='/kaggle/working/derm-classifier-v7-kfold-tta-cm.png')

# Salva as predicoes out-of-fold (uteis para figuras/analises no texto)
pd.DataFrame({
    'true_label': oof_y, 'pred_plain': oof_plain, 'pred_tta': oof_tta,
    'true_group': [LABEL_TO_GROUP[l] for l in oof_y],
    'pred_group_plain': [LABEL_TO_GROUP[p] for p in oof_plain],
    'pred_group_tta': [LABEL_TO_GROUP[p] for p in oof_tta],
}).to_csv('/kaggle/working/derm-classifier-v7-kfold/oof_predictions.csv', index=False)
print("\nSalvou matrizes (PNG) e oof_predictions.csv")